### Crie uma tabela de Pedidos (orders) unindo Itens do Pedido (order_items), Pedidos (orders), Equipes (staffs) e Lojas (stores).

Crie um case mapeando os Status abaixo: <br>
1 THEN 'Pending' - Pendente <br>
2 THEN 'Processing' - Processando <br> 
3 THEN 'Shipped' - Expedido <br>
4 THEN 'Delivered' - Entregue <br>
ELSE 'Unknown' - Desconhecido

In [0]:
%python
# Definir pastas do projeto em variáveis para facilitar
bronze_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/'
silver_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/silver/'
gold_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/gold/'
resource_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/origem'
resource_path_volume = '/Volumes/bikestore/logistics/bikestore_resource/origem/'

In [0]:
%python
# Criando várias tabelas temporárias 
bronze_map = {
    #'tmp_bronze_brands': f'{bronze_path}/brands/',
    #'tmp_bronze_categories': f'{bronze_path}/categories/',
    #'tmp_bronze_customers': f'{bronze_path}/customers/',
    'tmp_bronze_order_items': f'{bronze_path}/order_items/',
    'tmp_bronze_orders': f'{bronze_path}/orders/',
    #'tmp_bronze_products': f'{bronze_path}/products/',
    'tmp_bronze_staffs': f'{bronze_path}/staffs/',
    #'tmp_bronze_stocks': f'{bronze_path}/stocks/',
    'tmp_bronze_stores': f'{bronze_path}/stores/'
}
for view_name, path in bronze_map.items():
    (spark.read.format('delta')
    .load(path)
    .createOrReplaceTempView(view_name)
) 

In [0]:
%python
# Salvando no formato delta na camada Silver
df_orders_silver = spark.sql("""
-- Realizando o CTE
WITH order_items as (
    SELECT
    OI.order_id,
    OI.item_id,
    OI.product_id,
    OI.quantity,
    OI.list_price ,
    -- (OI.list_price * OI.quantity) AS value_test,
    ROUND((OI.list_price * OI.quantity) * (1-OI.discount), 2)AS total_sale,
    OI.discount
FROM tmp_bronze_order_items AS OI
)

-- Select final
SELECT
    ORD.order_id,
    ORD.customer_id,
    CASE 
        WHEN ORD.order_status = 1 THEN 'Pending'
        WHEN ORD.order_status = 2 THEN 'Processing'
        WHEN ORD.order_status = 3 THEN 'Shipped'
        WHEN ORD.order_status = 4 THEN 'Delivered'
        ELSE 'Unknown'
    END AS status,
    ORD.order_status,
    ORD.order_date,
    ORD.required_date,
    ORD.shipped_date,
    -- ORD.store_id,
    -- ST.store_id AS store_id_loja,
    ST.store_name,
    ST.state,
    ST.city,
    -- ORD.staff_id,
    -- STF.staff_id AS staff_id_vendedor,
    STF.first_name AS staff_nome,
    STF.active AS active_staff,
    STF.email,
    OIT.product_id,
    OIT.quantity,
    OIT.total_sale,
    OIT.list_price,
    OIT.discount
FROM tmp_bronze_orders AS ORD
LEFT JOIN    tmp_bronze_stores AS ST ON ORD.store_id = ST.store_id
LEFT JOIN   tmp_bronze_staffs AS STF ON ORD.staff_id = STF.staff_id
LEFT JOIN         order_items AS OIT ON ORD.order_id = OIT.order_id
""")

# Salvar e transformar em formato delta na camada Silver
df_orders_silver.write \
    .mode('overwrite') \
    .format('delta') \
    .option('mergeSchema', 'true') \
    .save(f'{silver_path}/orders')


In [0]:
/*Criar tabela física de produção, este comando precisa rodar só 1 vez, pois quando os arquivos forem atualizados a tabela refletirá o resultado (Vai obter o log mais recente e trazer dados do(s) arquivo(s) atualizado)
CREATE TABLE IF NOT EXISTS bikestore.logistics.silver_orders
LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/silver/orders';*/

In [0]:
-- SELECT * FROM bikestore.logistics.silver_product LIMIT 15